# This function calculates the water balance for a watershed using:
* Shapefile containing the watershed domain
* Precipitation from Stage-IV (or Daymet)
* Streamflow observations at the watershed outlet
* Total ET from MODIS


In [301]:
# Import packages
import sys, os
import h5py
import pandas as pd
import geopandas as gpd
import numpy as np
import netCDF4 as nc
import fiona
import matplotlib.dates as mdates
from matplotlib import pyplot as plt
from matplotlib import cm as pcm
import ipympl
# %matplotlib ipympl # This line is for interactive plots
# %matplotlib inline
import matplotlib.path as mplp

import matplotlib
import watershed_workflow 
from pyproj import Proj, transform
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import xarray as xr

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
import cartopy.mpl.ticker as cticker


# import hydrofunctions as hf
import datetime
import time

from datetime import datetime
import os

Define global parameters

In [302]:
# Read the USGS ID for all the gages in the Neches River Basin (see get_basins_streamflow_neches.ipynb)
list_gauges = pd.read_csv('../data-processed/basin/List_gaugeID.csv')
list_gauges = list_gauges['gauge_ID'] 
# Add a zero at the beginning of the gauge ID
list_gauges = ['0' + str(i) for i in list_gauges]

Site = list_gauges[11]

In [303]:
# Set time interval to perform water balance
start_date = '2010-01-01'
end_date = '2019-12-30'

Check_rain_pattern = False
# 'StageIV' or 'Daymet'. Note: StageIV can be used for the entire CONUS. For Daymet, the data most be downloaded using the ATS repo

if Check_rain_pattern:
    Buffer_bd = 8 # buffer for Stage IV. [degree]
    # Use a larger buffer (Buffer_bd = 8)to check that the final storm pattern agrees with maps from mesonet 
    # http://mesonet.agron.iastate.edu/docs/nexrad_composites/
else:
    Buffer_bd = 0.05 

# Read watershed and get exterior coordinates
Path_basin=f'../data-processed/basin/watershed_gaugeID_{Site}.shp' # the shapefile is in 'epsg:4326'
basin = gpd.read_file(Path_basin)
crs_Daymet = watershed_workflow.crs.daymet_crs()
basin_daymet = basin.to_crs(crs_Daymet)
g = [i for i in basin_daymet.geometry]
x,y = g[0].exterior.coords.xy
coor_basin_boundary = np.column_stack((x,y))

## Read and display DayMet data

In [304]:
# Read ATS-DayMet data
# This is daily data and it starts from 01-01-start_Year and it ends in 12-31-end_Year
Path_example_ATS= '../../../global_data/rainfall_input_ATS/Neches/Neches_DayMet_2000_2020.h5'
# Path_example_ATS= '/Users/gpq/ORNL/Projects/SETx_Urban/Jupyter_Notebooks/watershed_workflow/Village_creek_tin/data_processed/Village_gauge_T_DayMet_1980_2020.h5'
DayMet_Data = h5py.File(Path_example_ATS, 'r')
DayMet_Data.keys()


<KeysViewHDF5 ['air temperature [K]', 'incoming shortwave radiation [W m^-2]', 'precipitation rain [m s^-1]', 'precipitation snow [m SWE s^-1]', 'relative humidity [-]', 'time [s]', 'x [m]', 'y [m]']>

In [305]:
# Create a Mask
# Points representing the pixel location in the grid
X_points=DayMet_Data['x [m]']
Y_points=DayMet_Data['y [m]']
Time_ATS=DayMet_Data['time [s]']
xv, yv = np.meshgrid(X_points, Y_points)
points = np.array((xv.flatten(), yv.flatten())).T
mpath = mplp.Path(coor_basin_boundary)
Mask = mpath.contains_points(points).reshape(xv.shape)

# Check that number Time_ATS length is the same that the expected Total_days from 2010 to 2020
Total_days = ((pd.Timestamp("2020-12-30").year-pd.Timestamp("2000-01-01").year)+1)*365
print(Time_ATS.shape,"=",Total_days)


(7665,) = 7665


In [306]:
Y_points[:]

array([-1333000., -1332000., -1331000., -1330000., -1329000., -1328000.,
       -1327000., -1326000., -1325000., -1324000., -1323000., -1322000.,
       -1321000., -1320000., -1319000., -1318000., -1317000., -1316000.,
       -1315000., -1314000., -1313000., -1312000., -1311000., -1310000.,
       -1309000., -1308000., -1307000., -1306000., -1305000., -1304000.,
       -1303000., -1302000., -1301000., -1300000., -1299000., -1298000.,
       -1297000., -1296000., -1295000., -1294000., -1293000., -1292000.,
       -1291000., -1290000., -1289000., -1288000., -1287000., -1286000.,
       -1285000., -1284000., -1283000., -1282000., -1281000., -1280000.,
       -1279000., -1278000., -1277000., -1276000., -1275000., -1274000.,
       -1273000., -1272000., -1271000., -1270000., -1269000., -1268000.,
       -1267000., -1266000., -1265000., -1264000., -1263000., -1262000.,
       -1261000., -1260000., -1259000., -1258000., -1257000., -1256000.,
       -1255000., -1254000., -1253000., -1252000., 

In [307]:
# Convert the DayMet data to a handable structure for xarray
Dimensions = np.array(DayMet_Data['precipitation rain [m s^-1]']['0'].shape)
rain_matrix = np.zeros(shape=(Dimensions[0],Dimensions[1],len(DayMet_Data['precipitation rain [m s^-1]'].keys())))

for key in DayMet_Data['precipitation rain [m s^-1]'].keys():
    rain_matrix[:,:,int(key)] = np.array(DayMet_Data['precipitation rain [m s^-1]'][key][:,:])


In [308]:
# Select the data for a given time interval

Time_pd_ATS = pd.date_range(pd.Timestamp("2000-01-01"), pd.Timestamp("2020-12-31"), freq="24H")
# Remove 31-Dec in leap years, since that DayMet format remove those to keep 365 days
Time_pd_ATS = Time_pd_ATS[Time_pd_ATS.day_of_year!=366]
Time_pd_ATS.shape
# Time interval of the rainfall event
Start_event = pd.Timestamp(start_date)
End_event = pd.Timestamp(end_date)
Time_pd_ATS_event = pd.date_range(Start_event, End_event, freq="24H")
# Remove 31-Dec in leap years, since that DayMet format remove those to keep 365 days
Time_pd_ATS_event = Time_pd_ATS_event[Time_pd_ATS_event.day_of_year!=366]
mask = (Time_pd_ATS >= Start_event) & (Time_pd_ATS <= End_event)
rain_matrix_event = rain_matrix[:,:,mask]


In [309]:
# Accumulate rainfall
avgrain=np.nansum(rain_matrix_event*24*60*60,axis=2)*1000 # from m to mm

# Use mask to remove data outside the watershed
rain_matrix_event[Mask==np.zeros(Mask.shape),:]=np.nan
# Compute total preci for the watershed along time
avgrain_time = np.nanmean(rain_matrix_event*24*60*60,axis=(0,1))*1000 # from m to mm. [mm/day]
# Transform to [mm/hr]
df_daily = pd.DataFrame({'Date': Time_pd_ATS_event, 'mm/day': avgrain_time})
df_daily = df_daily.set_index('Date')
df_daily['mm/hr'] = df_daily['mm/day']/24 
df_hourly = df_daily.resample('H').ffill()
Total_rainfall_basin = np.nansum(avgrain_time)
# put data into a dataset
xplot_avgrain=xr.Dataset(
        data_vars=dict(avgrain=(["y","x"],avgrain)),
        coords=dict(
            lat=(["y"],Y_points),
            lon=(["x"],X_points)),
        attrs=dict(description="Total precipitation [mm]"),
    )

In [310]:
df_hourly

,mm/day,mm/hr
Date,,
2010-01-01 00:00:00,0.000000,0.000000
2010-01-01 01:00:00,0.000000,0.000000
2010-01-01 02:00:00,0.000000,0.000000
2010-01-01 03:00:00,0.000000,0.000000
2010-01-01 04:00:00,0.000000,0.000000
...,...,...
2019-12-29 20:00:00,7.357244,0.306552
2019-12-29 21:00:00,7.357244,0.306552
2019-12-29 22:00:00,7.357244,0.306552


# Read and display Stage-IV data.

In [311]:
# Stage-IV data is taken fromo RainyDay input files, which it has a cooridnate transformation to obatin a regular rectangular mesh
# This data is processed in epsg:4326 crs

Path_example_StageIV = '/Users/gpq/ORNL/Databases/StageIV_RainyDay/StageIV_FilledCorr.20061019.03degree.nc'
# ds = xr.open_dataset(Path_example_StageIV)
# df = ds.to_dataframe()


In [312]:
# Get initial data including, basin bounds, and coordinate vectors
basin_bounds = basin.bounds 
ds = nc.Dataset(Path_example_StageIV)
Time_StageIV = ds['time']
lon_StageIV = ds['longitude'][:]
lat_StageIV = ds['latitude'][:]
# RainyDay coordinates are for cell upper left corner. These are translated to the center
deltax = (lon_StageIV[1] - lon_StageIV[0])/2
deltay = (lat_StageIV[1] - lat_StageIV[0])/2
lon_StageIV = lon_StageIV + deltax
lat_StageIV = lat_StageIV + deltay
# coordinates basin exterior
g = [i for i in basin.geometry]
x,y = g[0].exterior.coords.xy
coor_basin_boundary = np.column_stack((x,y))

In [313]:
# # Do transfomration to DayMet crs
# # crs_Daymet = watershed_workflow.crs.daymet_crs()
# inProj = Proj(init='epsg:4326')
# # lonv_StageIV_new,latv_StageIV_new = transform(inProj,crs_Daymet,lonv_StageIV.flatten(),latv_StageIV.flatten())
# lonv_StageIV_new,latv_StageIV_new = transform(inProj,inProj,lonv_StageIV.flatten(),latv_StageIV.flatten())
# lonv_StageIV_new = lonv_StageIV_new.reshape(lonv_StageIV.shape)
# latv_StageIV_new = latv_StageIV_new.reshape(latv_StageIV.shape)
# lon_StageIV_new = lonv_StageIV_new[0,:]
# lat_StageIV_new = latv_StageIV_new[:,0]


In [314]:
# Clip the data with the watershed bounds
lon_range_clip = (lon_StageIV <= (basin_bounds.maxx[0] + Buffer_bd)) & (lon_StageIV >= (basin_bounds.minx[0] - Buffer_bd))
lat_range_clip = (lat_StageIV <= (basin_bounds.maxy[0] + Buffer_bd)) & (lat_StageIV  >= (basin_bounds.miny[0] - Buffer_bd))
lon_StageIV_new = lon_StageIV[lon_range_clip]
lat_StageIV_new = lat_StageIV[lat_range_clip]


In [315]:
Start_event = pd.Timestamp(f'{start_date} 00:00:00')
End_event = pd.Timestamp(f'{end_date} 23:00:00')
Time_pd_StageIV_event = pd.date_range(Start_event, End_event, freq="H")

rain_matrix_StageIV = np.zeros(shape=(len(Time_pd_StageIV_event),len(lat_StageIV_new),len(lon_StageIV_new))) # Time, lat, lon
rain_matrix_StageIV[0:24,:,:] = np.array(ds['rainrate'][:,lat_range_clip,lon_range_clip])


In [316]:
# print(Time_pd_StageIV_event.year, Time_pd_StageIV_event.month, Time_pd_StageIV_event.day)
Date_Files = pd.date_range(Start_event, End_event, freq="24H")
Date_Files = Date_Files.strftime('%Y%m%d')
Date_Files[0]

# Get each file
loc_start = 0 
loc_end = 24
for key_file in Date_Files:
    Path_StageIV= f'/Users/gpq/ORNL/Databases/StageIV_RainyDay/StageIV_FilledCorr.{key_file}.03degree.nc'
    # check if file exists
    if os.path.exists(Path_StageIV):
        ds = nc.Dataset(Path_StageIV)
        rain_matrix_StageIV[loc_start:loc_end,:,:] = np.array(ds['rainrate'][:,lat_range_clip,lon_range_clip])
    else:
        rain_matrix_StageIV[loc_start:loc_end,:,:] = 0
    loc_start = loc_start + 24
    loc_end = loc_end + 24



In [317]:
# # Create a mask !!!
# xv_StageIV_new,yv_StageIV_new = transform(inProj,crs_Daymet,xv_StageIV.flatten(),yv_StageIV.flatten())

latv_StageIV, lonv_StageIV = np.meshgrid(lat_StageIV_new, lon_StageIV_new)
points = np.array((lonv_StageIV.flatten(), latv_StageIV.flatten())).T
mpath = mplp.Path(coor_basin_boundary)
Mask_StageIV = mpath.contains_points(points).reshape((len(lon_StageIV_new),len(lat_StageIV_new)))

In [318]:
Mask_StageIV.T.shape

(88, 72)

In [319]:
# Do plot
# Accumulate rainfall
avgrain_StageIV=np.nansum(rain_matrix_StageIV*1,axis=0) # mm
# Use mask to remove data outside the watershed
rain_matrix_StageIV[:,Mask_StageIV.T==np.zeros(Mask_StageIV.T.shape)]=np.nan
# Compute total preci for the watershed along time
avgrain_time_StageIV = np.nanmean(rain_matrix_StageIV*1,axis=(1,2)) # mm
Total_rainfall_basin_StageIV = np.nansum(avgrain_time_StageIV)

In [320]:
states_provinces = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_1_states_provinces_lines',
        scale='50m',
        facecolor='none')
coast_10m = cfeature.NaturalEarthFeature("physical", "land", "10m", edgecolor="k", facecolor="0.8")

In [321]:
avgrain_StageIV[avgrain_StageIV == 0.0] = np.nan
# put data into a dataset
xplot_avgrain_StageIV=xr.Dataset(
        data_vars=dict(avgrain=(["y","x"],avgrain_StageIV)),
        coords=dict(
            lat=(["y"],lat_StageIV_new),
            lon=(["x"],lon_StageIV_new)),
        attrs=dict(description="Total precipitation [mm]"),
)

Number_Days=(End_event-Start_event).days+1

## Do a nice plot showing spatial rainfall and hietographs

In [322]:
# Plot showing the large spatial pattern (buffer = 4) of rainfall from Stage-IV. This most agree with the mesonet images
if Check_rain_pattern:
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(25,7))
    ax1 = fig.add_subplot(1,1,1,projection=proj)
    xplot_avgrain_StageIV.avgrain.plot(x="lon",y="lat",yincrease=True,cmap='Blues',cbar_kwargs={'orientation':'vertical','label':f"Total precipitation [mm] over {Number_Days} days" }, ax=ax1)
    # vmin=500, vmax=1000,
    ax1.set(xlabel=None,ylabel=None)
    ax1.add_feature(states_provinces)
    ax1.add_feature(coast_10m, edgecolor='black', facecolor='none')
    basin.plot(edgecolor='red', facecolor='none',ax=ax1)
    ax1.set_title(f'Stage-IV Precipitation\n from {Start_event} to {End_event}')


In [323]:
# Do a final plot comparing all the results. Plot showing the two spatial patterns, and the histogram at mm/hr scale
# Check the time reference of the two datasets
# %matplotlib ipympl 
# if Check_rain_pattern == False:
#     proj = ccrs.PlateCarree()
#     fig = plt.figure(figsize=(15,5))
#     ax1 = fig.add_subplot(1,3,1,projection=proj)
#     ax2 = fig.add_subplot(1,3,2)
#     ax3 = fig.add_subplot(1,3,3)

#     minRain = np.min([np.nanmin(avgrain[:]), np.nanmin(avgrain_StageIV[:])])
#     maxRain = np.max([np.nanmax(avgrain[:]), np.nanmax(avgrain_StageIV[:])])

#     # RainyDay
#     xplot_avgrain_StageIV.avgrain.plot(x="lon",y="lat",yincrease=True,cmap='Blues',cbar_kwargs={'orientation':'vertical','label':f"Total precipitation [mm] over {Number_Days} days" },
#                                         vmin=minRain, vmax=maxRain, ax=ax1)
#     ax1.set(xlabel=None,ylabel=None)
#     ax2.set_xticks([])
#     ax2.set_yticks([])
#     ax1.add_feature(states_provinces)
#     ax1.add_feature(coast_10m, edgecolor='black', facecolor='none')
#     basin.plot(edgecolor='red', facecolor='none',ax=ax1)
#     ax1.title.set_text(f'Stage-IV Precipitation\n from {Start_event} to {End_event} \n Total Precipitation over the basin = {round(Total_rainfall_basin_StageIV)} mm')
#     ax1.set_box_aspect(1)

#     # DayMet
#     xplot_avgrain.avgrain.plot(x="lon",y="lat",yincrease=True,cmap='Blues',cbar_kwargs={'orientation':'vertical','label':f"Total precipitation [mm] over {Number_Days} days" }, 
#                             vmin=minRain, vmax=maxRain, ax=ax2)
#     basin_daymet.plot(edgecolor='red', facecolor='none',ax=ax2)
#     # ax[0].set_title(f'DayMet Precipitation\n from {Start_event} to {End_event}')
#     ax2.set(xlabel=None,ylabel=None)
#     ax2.set_xticks([])
#     ax2.set_yticks([])
#     ax2.title.set_text(f'DayMet Precipitation\n from {Start_event} to {End_event} \n Total Precipitation over the basin = {round(Total_rainfall_basin)} mm')
#     ax2.set_box_aspect(1)

#     # Histograms
#     ax3.bar(Time_pd_StageIV_event,avgrain_time_StageIV, alpha = 0.5, width = 0.05)
#     ax3.bar(df_hourly.index,df_hourly['mm/hr'], alpha = 0.5, width = 0.05)
#     ax3.tick_params(axis='x', rotation=45)
#     ax3.set(xlabel=None,ylabel='Average rainfall intensity over the basin [mm/hour]')
#     ax3.legend(['Stage-IV', "DayMet"])


In [324]:
df_Daymet = df_hourly.copy()

# Remove first column and rename last column
df_Daymet = df_Daymet.iloc[:,1:]
df_Daymet = df_Daymet.rename(columns={'mm/hr':'Rainfall [mm/hr]'})
df_Daymet

,Rainfall [mm/hr]
Date,
2010-01-01 00:00:00,0.000000
2010-01-01 01:00:00,0.000000
2010-01-01 02:00:00,0.000000
2010-01-01 03:00:00,0.000000
2010-01-01 04:00:00,0.000000
...,...
2019-12-29 20:00:00,0.306552
2019-12-29 21:00:00,0.306552
2019-12-29 22:00:00,0.306552


In [325]:
# Create dataframe with the rainfall and date for Stage-IV and Daymet

df_StageIV = pd.DataFrame(data=avgrain_time_StageIV, index=Time_pd_StageIV_event, columns=['Rainfall [mm/hr]'])
df_Daymet = df_hourly.copy()
# Remove first column and rename last column
df_Daymet = df_Daymet.iloc[:,1:]
df_Daymet = df_Daymet.rename(columns={'mm/hr':'Rainfall [mm/hr]'})

# Export data to csv
df_StageIV.to_csv(f'../../../global_data/rainfall_input_ATS/Neches/Basin_{Site}_StageIV_2000_2020.csv')
df_Daymet.to_csv(f'../../../global_data/rainfall_input_ATS/Neches/Basin_{Site}_Daymet_2000_2020.csv')
